# Multi-Echelon Automotive Supply Chain — Analytic Notebook

**Course capstone · Knowledge graph analysis (Neo4j + data science)**

**Students:** Bekithemba Nkomo · Masheia Dzimba · Peter Mangoro

---

This notebook **loads the 2020 automotive production network** from the Moetz et al. (2020) workbook ([Mendeley Data](https://doi.org/10.17632/pr3sdy5vp3.1)), performs **graph-style exploratory analysis in Python** (pandas + NetworkX), and documents **Cypher + GDS** workflows you can run in Neo4j Desktop against the same model.

**Why Python here?** Jupyter is ideal for reproducible tables, plots, and narrative. Neo4j Browser is ideal for interactive Cypher. This notebook combines **executable analytics on the raw sheets** with **copy-paste Cypher** for your populated database.

---


## 0. Environment

Install dependencies once (from the project folder):

```bash
pip install -r requirements.txt
```

**Data path:** place `2020_dataset_OfAutomotiveProductionNetwork.xlsb` inside the `Data for Optimisation model...` folder (as in your repo), or set `DATA_PATH` below.


In [ ]:
%matplotlib inline
from __future__ import annotations

import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
from pyxlsb import open_workbook

warnings.filterwarnings("ignore", category=FutureWarning)

try:
    from IPython.display import display
except ImportError:
    display = print  # noqa: A001

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["figure.dpi"] = 110

# Project folder = notebook folder (change if you open the notebook from elsewhere)
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / (
    "Data for Optimisation model for multi-item multi-echelon supply chains "
    "with nested multi-level products"
)
DATA_PATH = DATA_DIR / "2020_dataset_OfAutomotiveProductionNetwork.xlsb"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Expected workbook at:\n  {DATA_PATH}\nSet DATA_PATH to your .xlsb location."
    )
print("Using:", DATA_PATH)


: 

In [ ]:
def read_xlsb_sheet(path: Path, sheet: str) -> pd.DataFrame:
    """Read one sheet from Excel binary (.xlsb) into a DataFrame."""
    with open_workbook(path) as wb:
        with wb.get_sheet(sheet) as s:
            rows = [[item.v for item in r] for r in s.rows()]
    if not rows:
        return pd.DataFrame()
    header, body = rows[0], rows[1:]
    return pd.DataFrame(body, columns=header)


print("Loader ready.")


## 1. Load workbook tables

Sheets follow the published dataset: `products`, `nodes`, `arcs`, `BOM`, `demands`, capacity and flow tables, etc. First load may take **~1–2 minutes** on typical laptops because `BOM` and `demands` are large.


In [ ]:
SHEETS = [
    "products",
    "nodes",
    "nodes_inflow",
    "arcs",
    "capacity_at_arc",
    "max_flow_product_per_arc",
    "max_flow_group_per_arc",
    "operations",
    "BOM",
    "demands",
    "initial_inventories",
    "initial_flows",
]

tables: dict[str, pd.DataFrame] = {}
for name in SHEETS:
    print(f"Loading {name}...", end=" ")
    tables[name] = read_xlsb_sheet(DATA_PATH, name)
    print(f"{len(tables[name]):,} rows")

products = tables["products"].rename(
    columns={"product_p": "product_id", "group_g": "group", "transportation_size_s": "transport_size"}
)
nodes = tables["nodes"].rename(columns={"node_n": "node_id"})
arcs = tables["arcs"].rename(
    columns={
        "starting_node_i": "from_node",
        "ending_node_j": "to_node",
        "process_lead_time_l_ij": "lead_time",
        "group_g": "product_group",
    }
)
bom = tables["BOM"].rename(
    columns={
        "mother": "mother_id",
        "child": "child_id",
        "individual_input_quantity_q_mc": "qty",
    }
)
demands = tables["demands"].rename(
    columns={
        "node_n": "node_id",
        "product_p": "product_id",
        "demand_d_npt": "demand_qty",
        "period_t": "period",
    }
)
capacity_arc = tables["capacity_at_arc"].rename(
    columns={
        "starting_node_i": "from_node",
        "ending_node_j": "to_node",
        "period_t": "period",
        "capacity_c_ijt": "capacity",
    }
)
initial_inv = tables["initial_inventories"].rename(
    columns={
        "node_n": "node_id",
        "product_p": "product_id",
        "initial_inventory_I_np0": "initial_inv",
        "safety_stock": "safety_stock",
        "max_inventory": "max_inventory",
        "period_t": "period",
    }
)
initial_flows = tables["initial_flows"].rename(
    columns={
        "starting_node_i": "from_node",
        "ending_node_j": "to_node",
        "product_p": "product_id",
        "period_t": "period",
        "initial_flow": "flow_qty",
    }
)

print("\nCore frames: products, nodes, arcs, bom, demands — ready.")


## 2. Graph EDA (≥ 8 questions)

We answer the same questions you would ask after ingesting into Neo4j: counts, BOM structure, logistics degrees, demand coverage, inventory and flow concentration.


In [ ]:
# Q1 — How many nodes, products, groups, periods?
n_nodes = nodes["node_id"].nunique()
n_products = products["product_id"].nunique()
n_groups = products["group"].nunique()
periods_demand = demands["period"].dropna().unique()
periods_cap = capacity_arc["period"].dropna().unique()
n_periods_dem = len(np.unique(periods_demand))
n_periods_cap = len(np.unique(periods_cap))

summary = pd.DataFrame(
    {
        "metric": [
            "supply_chain_nodes",
            "unique_products",
            "product_groups",
            "distinct_demand_periods",
            "distinct_capacity_periods",
        ],
        "value": [n_nodes, n_products, n_groups, n_periods_dem, n_periods_cap],
    }
)
display(summary)


In [ ]:
# Q2 — Which product groups contain the most products?
group_counts = (
    products.groupby("group", as_index=False)
    .size()
    .rename(columns={"size": "n_products"})
    .sort_values("n_products", ascending=False)
)
display(group_counts)

fig, ax = plt.subplots()
sns.barplot(data=group_counts.head(12), x="n_products", y="group", ax=ax, hue="group", palette="viridis", legend=False)
ax.set_title("Products per group (top 12)")
plt.tight_layout()
plt.show()


In [ ]:
# Q3 — Which products have the most direct BOM lines (as mother)?
mother_deg = bom.groupby("mother_id").size().rename("n_bom_children").sort_values(ascending=False)
display(mother_deg.head(15).to_frame())

# Q4 — Deepest upstream chains for finished cars (mother = car SKU)
car_ids = set(products.loc[products["group"] == "car", "product_id"].astype(str))

# Build adjacency: mother -> child (assembly consumes child)
bom_pairs = bom[["mother_id", "child_id"]].drop_duplicates()
bom_pairs["mother_id"] = bom_pairs["mother_id"].astype(str)
bom_pairs["child_id"] = bom_pairs["child_id"].astype(str)
children_by_mother = bom_pairs.groupby("mother_id")["child_id"].apply(list).to_dict()


def max_depth_from(mother: str, memo: dict, stack: set | None = None) -> int:
    if stack is None:
        stack = set()
    if mother in memo:
        return memo[mother]
    if mother in stack:
        return 0
    kids = [str(k) for k in children_by_mother.get(mother, [])]
    if not kids:
        memo[mother] = 0
        return 0
    stack.add(mother)
    d = 1 + max(max_depth_from(c, memo, stack) for c in kids)
    stack.remove(mother)
    memo[mother] = d
    return d


memo = {}
depths = {m: max_depth_from(str(m), memo) for m in car_ids}
depth_series = (
    pd.Series(depths, name="bom_depth")
    .sort_values(ascending=False)
)
display(depth_series.head(15).to_frame())
print("Max BOM depth (car SKUs):", int(depth_series.max()))


In [ ]:
# Q5 — Node in/out degree on logistics graph (arcs table)
edges = arcs[["from_node", "to_node"]].drop_duplicates()
G_log = nx.DiGraph()
G_log.add_edges_from(edges.itertuples(index=False, name=None))

in_d = dict(G_log.in_degree())
out_d = dict(G_log.out_degree())
deg_df = pd.DataFrame(
    {
        "node_id": list(G_log.nodes()),
        "in_degree": [in_d[n] for n in G_log.nodes()],
        "out_degree": [out_d[n] for n in G_log.nodes()],
    }
).sort_values(["in_degree", "out_degree"], ascending=False)
display(deg_df)


In [ ]:
# Q6 — Demand records per node
dem_by_node = demands.groupby("node_id").size().rename("n_demand_rows").sort_values(ascending=False)
display(dem_by_node.to_frame())

# Q7 — Number of periods in which each product is demanded
periods_per_product = (
    demands.groupby("product_id")["period"].nunique().rename("n_periods_demanded").sort_values(ascending=False)
)
display(periods_per_product.head(15).to_frame())


In [ ]:
# Q8 — Largest opening inventory and initial flows by node / arc
inv_by_node = (
    initial_inv.groupby("node_id")["initial_inv"].sum().sort_values(ascending=False).rename("sum_initial_inv")
)
display(inv_by_node.to_frame())

flow_sum = (
    initial_flows.groupby(["from_node", "to_node"])["flow_qty"]
    .sum()
    .sort_values(ascending=False)
    .rename("sum_initial_flow")
)
display(flow_sum.head(15).to_frame())


## 3. Deeper analysis A — BOM complexity for finished vehicles

**Question:** Which finished products sit on the **deepest** or **broadest** upstream component chains?

Below: top cars by **BOM depth** (already computed) and by **number of distinct components** (transitive where feasible — here we approximate with direct children count and max depth).


In [ ]:
bom_breadth = bom.groupby("mother_id").agg(n_direct_children=("child_id", "nunique")).sort_values(
    "n_direct_children", ascending=False
)
top_cars_breadth = bom_breadth.loc[bom_breadth.index.astype(str).isin(car_ids)].head(15)
display(top_cars_breadth)

# Shared components: how many car SKUs use each child?
child_usage = bom[bom["mother_id"].astype(str).isin(car_ids)].groupby("child_id").size().sort_values(ascending=False)
display(child_usage.head(15).to_frame(name="n_car_mothers"))


## 4. Deeper analysis B — Arc criticality (lead time × capacity stress)

**Question:** Which arcs are most exposed when **minimum per-period capacity** is tight relative to typical naming?

We summarize each arc: **max lead time** (from `arcs`), **minimum capacity** across periods (bottleneck periods), and **mean capacity**.


In [ ]:
arc_keys = arcs[["from_node", "to_node", "lead_time"]].drop_duplicates()
cap_agg = (
    capacity_arc.groupby(["from_node", "to_node"])
    .agg(min_capacity=("capacity", "min"), mean_capacity=("capacity", "mean"))
    .reset_index()
)
arc_stress = arc_keys.merge(cap_agg, on=["from_node", "to_node"], how="left")
# Simple composite score: higher lead time + lower min capacity => higher score
arc_stress["stress_score"] = arc_stress["lead_time"].fillna(0) / (arc_stress["min_capacity"].replace(0, np.nan))
arc_stress = arc_stress.sort_values("stress_score", ascending=False)
display(arc_stress)

fig, ax = plt.subplots()
sc = ax.scatter(
    arc_stress["lead_time"],
    arc_stress["min_capacity"],
    s=80,
    c=arc_stress["stress_score"].fillna(0),
    cmap="Reds",
    alpha=0.85,
)
plt.colorbar(sc, ax=ax, label="stress score (lead / min cap)")
ax.set_xlabel("Lead time (periods)")
ax.set_ylabel("Min per-period capacity (units)")
ax.set_title("Arc lead time vs minimum capacity")
plt.tight_layout()
plt.show()


## 5. GDS-style analysis — Betweenness centrality (logistics graph)

**Neo4j GDS:** project `Node` and `SHIPS_TO` / `FLOWS_TO`, then run **betweenness centrality**.

**Python equivalent:** directed graph from unique `(from_node, to_node)` in `arcs`, then NetworkX `betweenness_centrality` (unweighted). This identifies nodes that lie on many shortest paths — useful **bottleneck proxies** in a sparse logistics network.

> In Neo4j, use `gds.betweenness.stream` after `gds.graph.project` on your supply-chain node IDs.


In [ ]:
# Reuse G_log from Q5
between = nx.betweenness_centrality(G_log, normalized=True)
bet_df = (
    pd.DataFrame({"node_id": list(between.keys()), "betweenness": list(between.values())})
    .sort_values("betweenness", ascending=False)
    .reset_index(drop=True)
)
display(bet_df)

fig, ax = plt.subplots()
sns.barplot(data=bet_df, x="betweenness", y="node_id", ax=ax, hue="node_id", palette="mako", legend=False)
ax.set_title("Betweenness centrality (logistics digraph, unweighted)")
plt.tight_layout()
plt.show()

# Optional: draw network if small enough
pos = nx.spring_layout(G_log, seed=42, k=0.85)
plt.figure(figsize=(10, 7))
nx.draw_networkx_nodes(G_log, pos, node_size=900, node_color="#4ECDC4", alpha=0.9)
nx.draw_networkx_labels(G_log, pos, font_size=8)
nx.draw_networkx_edges(G_log, pos, arrows=True, arrowsize=20, edge_color="#555")
plt.title("Supply-chain logistics graph (arcs)")
plt.axis("off")
plt.tight_layout()
plt.show()


## 6. Neo4j: sample Cypher (run in Browser after import)

Use your actual property names; the reference doc maps sheets to relationship types.

**Constraints & counts**
```cypher
MATCH (n) RETURN labels(n)[0] AS label, count(*) AS c;
```

**Nodes with most outbound logistics relationships**
```cypher
MATCH (a)-[r:FLOWS_TO]->(b)
RETURN a.id AS from_node, count(*) AS out_arcs
ORDER BY out_arcs DESC;
```

**BOM depth (illustrative — depends on your Product model)**
```cypher
MATCH (p:Product {group: 'car'})-[:REQUIRES*1..20]->(c:Product)
RETURN p.id AS car, max(length(shortestPath((p)-[:REQUIRES*]->(c)))) AS depth
LIMIT 20;
```

**GDS betweenness (Neo4j 5 + GDS)**
```cypher
CALL gds.graph.drop('supply_net', false) YIELD graphName;
CALL gds.graph.project('supply_net', 'SupplyNode', 'FLOWS_TO') YIELD graphName, nodeCount, relationshipCount;
CALL gds.betweenness.stream('supply_net')
YIELD nodeId, score
RETURN gds.util.asNode(nodeId).id AS node_id, score
ORDER BY score DESC;
```

Adjust labels (`SupplyNode`, `FLOWS_TO`) to match your ingestion.


## 7. Optional — Neo4j Python driver

Set environment variables `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD` (or edit the block below). If unset, the cell skips without error.


In [ ]:
from neo4j import GraphDatabase

URI = os.environ.get("NEO4J_URI", "")
USER = os.environ.get("NEO4J_USER", "neo4j")
PASSWORD = os.environ.get("NEO4J_PASSWORD", "")


def run_cypher(uri: str, user: str, password: str, query: str):
    driver = GraphDatabase.driver(uri, auth=(user, password))
    with driver.session() as session:
        return session.run(query).data()


if URI and PASSWORD:
    sample = run_cypher(URI, USER, PASSWORD, "RETURN 1 AS ok")
    print("Neo4j OK:", sample)
else:
    print("Skip Neo4j driver demo — set NEO4J_URI and NEO4J_PASSWORD to connect.")


## References

- Moetz, A., Quetschlich, M., & Otto, B. (2020). *Data for: Optimisation model for multi-item multi-echelon supply chains with nested multi-level products* (Version 1). Mendeley Data. https://doi.org/10.17632/pr3sdy5vp3.1  
- Neo4j Graph Data Science: https://neo4j.com/docs/graph-data-science/current/
